# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading, exploring, and processing the [FAIR² mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. You can adapt and extend this analysis to suit your research, especially for protein abundance analysis and extracellular vesicle study.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Loaded Dataset: {}".format(metadata.name))
print("Description: {}".format(metadata.description))

# Optional: See publish date
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets and their fields, referencing all by their `@id` fields as required by the Croissant schema.

In [ ]:
# List all available record sets by their @id and name
record_set_objs = getattr(metadata, 'recordSets', None)
if record_set_objs is None:
    # For Croissant v1, sometimes `recordSet` is singular and inside a list
    record_set_objs = getattr(metadata, 'recordSet', [])

print("Available record sets:")
record_set_ids = []
for rs in record_set_objs:
    rid = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)
    name = rs['name'] if isinstance(rs, dict) and 'name' in rs else getattr(rs, 'name', None)
    print(f"  @id: {rid} | name: {name}")
    record_set_ids.append(rid)

# Choose a record set to explore in detail (here: the first one)
example_record_set_id = record_set_ids[0] if record_set_ids else None
print(f"\nExploring fields for record set: {example_record_set_id}")

record_set_obj = None
for rs in record_set_objs:
    if (isinstance(rs, dict) and rs['@id'] == example_record_set_id) or (hasattr(rs, '@id') and getattr(rs, '@id') == example_record_set_id):
        record_set_obj = rs
        break

# List field @id and name for this record set
if record_set_obj:
    fields = record_set_obj['field'] if isinstance(record_set_obj, dict) and 'field' in record_set_obj else getattr(record_set_obj, 'field', [])
    field_ids = []
    print("Fields in selected record set:")
    for field in fields:
        fid = field['@id'] if isinstance(field, dict) else getattr(field, '@id', None)
        fname = field['name'] if isinstance(field, dict) and 'name' in field else getattr(field, 'name', None)
        print(f"  @id: {fid} | name: {fname}")
        field_ids.append(fid)
else:
    print("No record set found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract all records into pandas DataFrames by @id
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

if example_record_set_id in dataframes:
    print("\nAvailable columns in example record set DataFrame:")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()
else:
    print("No records loaded for example record set.")

## 4. Exploratory Data Analysis (EDA)

Apply common EDA steps: filtering, normalization, and grouping. Here, we'll demonstrate by choosing a representative numeric field and, if available, a categorical field to group by. Remember: reference all fields by their `@id`!

**You may need to check the output from the previous code cell to find the appropriate field `@id` to use.**

*Example field suggestions:* If your dataset includes fields like `'coverage_percentage'`, `'peptide_counts'`, or `'MW_kDa'`, these are likely numeric. If `'accession'` or `'description'` exist, they are good for grouping.

In [ ]:
# Set your field IDs according to what you see in columns above
numeric_field_id = None
group_field_id = None
cols = dataframes[example_record_set_id].columns.tolist() if example_record_set_id in dataframes else []

# Guess likely numeric and group fields
for col in cols:
    if numeric_field_id is None and ("coverage" in col or "MW" in col or "peptide" in col or "abundance" in col):
        numeric_field_id = col
    if group_field_id is None and ("accession" in col or "description" in col or "sample" in col):
        group_field_id = col

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

df = dataframes[example_record_set_id]

# Ensure numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove outliers: e.g., those above the 99th percentile
upper = df[numeric_field_id].quantile(0.99)
filtered_df = df[df[numeric_field_id] < upper]
print(f"After outlier removal (numeric field < {upper:.2f}), data size: {len(filtered_df)}")

# Filtering: Keep only values above median
threshold = filtered_df[numeric_field_id].median()
filtered_df2 = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered down to {len(filtered_df2)} records with {numeric_field_id} > median ({threshold:.2f})")

# Normalization: zero mean, unit std
filtered_df2[f"{numeric_field_id}_normalized"] = (
    filtered_df2[numeric_field_id] - filtered_df2[numeric_field_id].mean()
) / filtered_df2[numeric_field_id].std()

print(filtered_df2[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id and show mean
if group_field_id is not None:
    grouped = filtered_df2.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGroup mean {numeric_field_id} by {group_field_id} (top 5):")
    print(grouped.head())

## 5. Visualization

Visualize the distribution of the selected numeric field, as well as the grouped means, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(filtered_df2[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

if group_field_id is not None:
    # Plot mean by groups (top 10)
    grouped = filtered_df2.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)[:10]
    plt.figure(figsize=(10, 4))
    sns.barplot(x=grouped.index, y=grouped.values)
    plt.title(f"Top 10 group means for {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded and explored the [FAIR² mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.
- Key record sets and fields were referenced by their `@id`, allowing for robust data extraction and transformation.
- After outlier removal and normalization, we visualized a numeric field (such as abundance or coverage %), grouped by a key attribute (e.g., protein accession or sample).

**Next steps:** Customize your field selection, filtering criteria, and visualizations for your scientific question (e.g., biomarker discovery, comparative analysis, or machine learning).